# บทที่ 11 - โปรโตคอลบริบทของโมเดล (MCP)

**โปรโตคอลบริบทของโมเดล (MCP)** เป็นมาตรฐานเปิดที่ช่วยให้เอเจนต์สามารถค้นหาและใช้เครื่องมือ แหล่งข้อมูล และแหล่งข้อมูลต่างๆ ได้แบบไดนามิกในระหว่างรันไทม์ แทนที่จะฝังเครื่องมือไว้ในเอเจนต์โดยตรง MCP ช่วยให้เอเจนต์เชื่อมต่อกับเซิร์ฟเวอร์ภายนอกที่เปิดเผยความสามารถตามความต้องการ

ในบทเรียนนี้ คุณจะได้เรียนรู้:
- MCP คืออะไรและเหตุใดจึงสำคัญกับระบบเอเจนต์
- สถาปัตยกรรมไคลเอนต์-เซิร์ฟเวอร์ของ MCP ทำงานอย่างไร
- วิธีสร้างเอเจนต์ที่ใช้การค้นหาเครื่องมือแบบ MCP


## การตั้งค่า

**ข้อกำหนดเบื้องต้น:**
- โครงการ Microsoft Foundry ที่มีการปรับใช้โมเดล
- รันคำสั่ง `az login` เพื่อทำการตรวจสอบสิทธิ์


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import dotenv
from typing import Annotated
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## โปรโตคอลบริบทโมเดล (MCP) คืออะไร?

MCP กำหนดวิธีมาตรฐานสำหรับเอเจนต์ AI ในการค้นหาและโต้ตอบกับเครื่องมือภายนอกและแหล่งข้อมูล:

- **MCP Server**: เปิดเผยเครื่องมือ, ทรัพยากร และพรอมต์ผ่านโปรโตคอลมาตรฐาน
- **MCP Client**: รันไทม์ของเอเจนต์ที่เชื่อมต่อกับเซิร์ฟเวอร์และค้นพบความสามารถที่มีอยู่
- **Dynamic Discovery**: เอเจนต์ไม่จำเป็นต้องมีเครื่องมือที่เข้ารหัสไว้ — พวกเขาค้นหาสิ่งที่มีอยู่ในเวลารันไทม์

นี่เป็นสิ่งที่ทรงพลังสำหรับการสร้างระบบเอเจนต์ที่ขยายได้ซึ่งสามารถเพิ่มความสามารถใหม่ๆ โดยไม่ต้องแก้ไขโค้ดของเอเจนต์


## วิธีการทำงานของ MCP

```
┌─────────────┐     discover      ┌─────────────────┐
│  MCP Client  │ ──────────────► │   MCP Server     │
│  (Agent)     │                  │  (Tool Provider) │
│              │ ◄────────────── │                   │
│              │   tool results   │  • list_tools()  │
│              │                  │  • call_tool()   │
└─────────────┘                  │  • resources     │
                                  └─────────────────┘
```

1. ตัวแทน (ลูกค้า MCP) เชื่อมต่อกับเซิร์ฟเวอร์ MCP
2. เซิร์ฟเวอร์ตอบกลับด้วยรายการเครื่องมือที่ใช้งานได้และโครงร่างของเครื่องมือเหล่านั้น
3. ตัวแทนสามารถเรียกใช้เครื่องมือที่ค้นพบระหว่างกระบวนการวิเคราะห์ของตนได้
4. ผลลัพธ์จะไหลกลับผ่านโปรโตคอลเดียวกัน


## การจำลองการค้นพบเครื่องมือ MCP

เนื่องจากเซิร์ฟเวอร์ MCP จริงต้องการกระบวนการเซิร์ฟเวอร์ที่กำลังทำงานอยู่ เราจะสาธิตรูปแบบโดยใช้ฟังก์ชัน `@tool` ที่จำลองสิ่งที่บริการที่พักที่เชื่อมต่อ MCP จะจัดหาให้

ในการใช้งานจริง เครื่องมือเหล่านี้จะถูกค้นพบแบบไดนามิกจากเซิร์ฟเวอร์ MCP แทนที่จะกำหนดไว้ในเครื่อง


In [ ]:
@tool(approval_mode="never_require")
def search_accommodations(
    location: Annotated[str, "The city to search for accommodations"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"] = 2
) -> str:
    """Search for accommodations (simulating an MCP-connected Airbnb tool).
    In production, this would be discovered via MCP from an accommodation service."""
    listings = {
        "Tokyo": [
            {"name": "Shinjuku Modern Apartment", "price": 120, "rating": 4.8},
            {"name": "Traditional Ryokan in Asakusa", "price": 200, "rating": 4.9},
            {"name": "Shibuya Studio", "price": 85, "rating": 4.5},
        ],
        "Paris": [
            {"name": "Le Marais Charming Flat", "price": 150, "rating": 4.7},
            {"name": "Montmartre Artist Loft", "price": 110, "rating": 4.6},
        ],
        "Barcelona": [
            {"name": "Gothic Quarter Penthouse", "price": 130, "rating": 4.8},
            {"name": "Barceloneta Beach Flat", "price": 95, "rating": 4.4},
        ],
    }
    results = listings.get(location, [])
    if not results:
        return f"No accommodations found in {location}"
    output = f"Accommodations in {location} ({check_in} to {check_out}, {guests} guests):\n"
    for listing in results:
        output += f"  - {listing['name']}: ${listing['price']}/night (★{listing['rating']})\n"
    return output


@tool(approval_mode="never_require")
def get_local_experiences(
    location: Annotated[str, "The city to find experiences in"],
    interest: Annotated[str, "Type of experience (food, culture, adventure, etc.)"] = "all"
) -> str:
    """Get local experiences and activities (simulating an MCP-connected tourism tool)."""
    experiences = {
        "Tokyo": {
            "food": ["Tsukiji Market Tour ($45)", "Ramen Making Class ($60)", "Sake Tasting ($35)"],
            "culture": ["Tea Ceremony ($50)", "Samurai Museum ($15)", "Sumo Tournament ($80)"],
            "adventure": ["Mt. Fuji Day Trip ($120)", "Go-kart City Tour ($80)"],
        },
        "Paris": {
            "food": ["Wine & Cheese Tasting ($55)", "Cooking Class ($90)", "Market Tour ($40)"],
            "culture": ["Louvre Guided Tour ($35)", "Montmartre Art Walk ($25)"],
        },
    }
    city_exp = experiences.get(location, {})
    if not city_exp:
        return f"No experiences found in {location}"
    if interest != "all" and interest in city_exp:
        items = city_exp[interest]
        return f"{interest.title()} experiences in {location}:\n" + "\n".join(f"  - {e}" for e in items)
    output = f"All experiences in {location}:\n"
    for cat, items in city_exp.items():
        output += f"\n  {cat.title()}:\n"
        for item in items:
            output += f"    - {item}\n"
    return output

## การสร้างเอเจนต์ด้วยเครื่องมือสไตล์ MCP


In [ ]:
agent = client.as_agent(
    tools=[search_accommodations, get_local_experiences],
    name="AccommodationAgent",
    instructions="""You are an accommodation and travel experiences specialist powered by MCP-connected services.

Help travelers find the perfect place to stay and things to do. When searching:
1. Use the search_accommodations tool to find listings
2. Use the get_local_experiences tool to suggest activities
3. Compare options and make personalized recommendations
4. Consider the traveler's budget, interests, and travel style""",
)

response = await agent.run(
    "I'm visiting Tokyo for 5 nights in April with my partner. We love traditional Japanese culture and food. "
    "Find us a place to stay and suggest some experiences.",
    )
print(response)

## MCP ในการผลิต

ในสภาพแวดล้อมการผลิต MCP ช่วยให้เกิดรูปแบบที่ทรงพลัง:

- **การค้นหาเครื่องมือแบบไดนามิก**: ตัวแทนเชื่อมต่อกับเซิร์ฟเวอร์ MCP และค้นหาเครื่องมือในเวลาทำงาน
- **สถาปัตยกรรมแบบแยกส่วน**: ผู้ให้บริการเครื่องมือสามารถอัปเดตได้อย่างอิสระจากตัวแทน
- **การแชร์ข้ามองค์กร**: ทีมสามารถเปิดเผยความสามารถผ่านเซิร์ฟเวอร์ MCP ที่ตัวแทนใดก็ได้สามารถใช้
- **การสนับสนุน Microsoft Agent Framework**: MAF มีการสนับสนุนลูกค้า MCP ในตัวผ่าน `mcp` integration

ในการใช้เซิร์ฟเวอร์ MCP จริงกับ MAF คุณจะเชื่อมต่อผ่าน `hosted_mcp_tool()` หรือการรวมลูกค้า MCP

**เรียนรู้เพิ่มเติม:**
- [ข้อกำหนด MCP](https://modelcontextprotocol.io/)
- [การสนับสนุน MCP ของ Microsoft Agent Framework](https://github.com/microsoft/agent-framework/tree/main/python/samples/02-agents/mcp)


## Summary

In this lesson, you learned:
- **MCP** is an open standard for dynamic tool discovery between agents and tool providers
- The **client-server architecture** lets agents discover capabilities at runtime
- MCP enables **extensible, decoupled agent systems** where tools can be added without code changes
- Microsoft Agent Framework provides **built-in MCP support** for production use


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ปฏิเสธความรับผิดชอบ**:
เอกสารนี้ได้รับการแปลโดยใช้บริการแปลภาษา AI [Co-op Translator](https://github.com/Azure/co-op-translator) ขณะที่เราพยายามให้ความถูกต้อง โปรดทราบว่าการแปลโดยอัตโนมัติอาจมีข้อผิดพลาดหรือความไม่ถูกต้อง เอกสารต้นฉบับในภาษาต้นทางควรถูกพิจารณาเป็นแหล่งข้อมูลที่เชื่อถือได้ สำหรับข้อมูลที่สำคัญ แนะนำให้ใช้การแปลโดยมนุษย์มืออาชีพ เราไม่รับผิดชอบต่อความเข้าใจผิดหรือการตีความที่ผิดพลาดที่เกิดขึ้นจากการใช้การแปลนี้
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
